# PDF → grafo → búsqueda semántica en Grafito

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jpmanson/GrafitoDB/blob/main/examples/semantic/pdf_chunking_colab.ipynb)

Pipeline mínimo sobre un PDF real de Anthropic —
**[Building Effective AI Agents](https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf)**:

PDF → markdown → `Document / Section / Chunk` → embeddings → `search` → `expand` + `pack`.

Dos decisiones de diseño de este notebook:

- **Sin conocer el documento.** No hardcodeamos los títulos del PDF: los encabezados
  se detectan por **tamaño de fuente** (la fuente más usada es el cuerpo; lo más grande,
  títulos). Funciona en cualquier PDF, y si uno no tiene jerarquía tipográfica el
  chunking cae a passages planos sin romperse.
- **Grafos legibles.** En vez de dibujar los cientos de chunks, mostramos el árbol de
  secciones y, por consulta, sólo el **vecindario** del hit.

> En Grafito **1 nodo = 1 vector**: un PDF largo son muchos passages enlazados, no un
> multi-vector en un solo nodo.


In [ ]:
%pip install -q "grafitodb[viz]>=0.4.0" pymupdf sentence-transformers networkx requests
print("OK")


## 1. Helpers de visualización


In [ ]:
from __future__ import annotations

import tempfile
from collections import Counter
from pathlib import Path
from typing import Any

import fitz  # PyMuPDF
import requests
from IPython.display import HTML, Markdown, display

from grafito import GrafitoDatabase
from grafito.document import DocumentIngestor, MarkdownChunker, TitleContextEnricher
from grafito.embedding_functions.base import EmbeddingFunction
from grafito.integrations import save_pyvis_html

COLORS = {"Document": "#264653", "DocumentVersion": "#2a9d8f",
          "Section": "#e9c46a", "Chunk": "#f4a261", "hit": "#e63946"}


def _label(node_id, attrs):
    p, labels = attrs.get("properties") or {}, attrs.get("labels") or []
    if "Document" in labels:
        return f"PDF: {p.get('title') or node_id}"[:40]
    if "DocumentVersion" in labels:
        return f"v{p.get('generation', '?')}"
    if "Section" in labels:
        return f"§ {(p.get('title') or '?')[:28]}"
    if "Chunk" in labels:
        return f"#{p.get('global_seq', '?')} {(p.get('text') or '')[:26]}…".replace(chr(10), " ")
    return str(node_id)


def show_graph(db, ids, *, title="", hits=None, height="460px"):
    """Draw only `ids` (a small subgraph). `hits` are painted red."""
    hits = hits or set()
    G = db.to_networkx().subgraph(ids).copy()
    for nid in G.nodes:
        a = G.nodes[nid]
        labels = a.get("labels") or []
        a["properties"] = {**(a.get("properties") or {}),
                           "_c": COLORS["hit"] if nid in hits else COLORS.get(labels[0] if labels else "", "#8ecae6")}
    path = Path(tempfile.gettempdir()) / "g.html"
    save_pyvis_html(G, path=str(path), notebook=False, directed=True, color_by_label=False,
                    node_color_attr="_c", label_fn=_label, physics="spread", height=height,
                    width="100%", bgcolor="#fff", font_color="#222", cdn_resources="in_line")
    if title:
        display(Markdown(f"### {title}"))
    display(HTML(f'<div style="border:1px solid #ddd;border-radius:8px">{path.read_text()}</div>'))


def legend():
    html = "".join(f'<span style="margin-right:14px"><span style="display:inline-block;width:11px;'
                   f'height:11px;background:{c};border-radius:50%;margin-right:5px"></span>{n}</span>'
                   for n, c in COLORS.items())
    display(HTML(f"<div style='font:14px sans-serif'>{html}</div>"))


legend()


## 2. PDF → markdown (encabezados por tamaño de fuente)

`fitz` (PyMuPDF) da el tamaño de cada línea: la fuente más frecuente es el cuerpo;
las líneas cortas con fuente mayor son títulos (`#` / `##` / `###`). **Sin títulos
hardcodeados** — el mismo código sirve para otro PDF.

Un PDF *diseñado* como este necesita dos limpiezas para no ensuciar la jerarquía:

- un **título que se parte en varias líneas** es un solo encabezado (se unen líneas
  consecutivas del mismo tamaño);
- una sección que aparece **dos veces** — un *banner* grande en la página divisoria
  más el encabezado real — se emite una sola vez (se conserva el más profundo).

Si un PDF no tiene jerarquía tipográfica, el chunking cae a passages planos sin romperse.


In [ ]:
PDF_URL = ("https://resources.anthropic.com/hubfs/Building%20Effective%20AI%20Agents-"
           "%20Architecture%20Patterns%20and%20Implementation%20Frameworks.pdf")
PDF_PATH = Path("agents.pdf")
PDF_PATH.write_bytes(requests.get(PDF_URL, timeout=120).content)


def pdf_to_markdown(path, *, max_title_len=90):
    """Texto plano de PDF -> markdown, con encabezados detectados por tamaño de fuente."""
    doc = fitz.open(str(path))
    sizes = Counter()
    for page in doc:
        for b in page.get_text("dict")["blocks"]:
            for line in b.get("lines", []):
                for s in line["spans"]:
                    sizes[round(s["size"])] += len(s["text"].strip())
    if not sizes:
        return ""
    body = sizes.most_common(1)[0][0]                         # tamaño dominante = cuerpo
    heads = sorted((s for s in sizes if s > body), reverse=True)[:3]
    level = {s: i + 1 for i, s in enumerate(heads)}           # el mayor -> nivel 1

    seq = []                                                  # (nivel, texto); nivel 0 = cuerpo
    for page in doc:
        for b in page.get_text("dict")["blocks"]:
            for line in b.get("lines", []):
                text = "".join(s["text"] for s in line["spans"]).strip()
                if not text:
                    continue
                size = round(max(s["size"] for s in line["spans"]))
                lv = level.get(size, 0)
                if lv and len(text) > max_title_len:
                    lv = 0                                    # demasiado largo para ser título
                if lv and seq and seq[-1][0] == lv:
                    seq[-1] = (lv, seq[-1][1] + " " + text)   # une un título partido en varias líneas
                else:
                    seq.append((lv, text))

    out = []                                                  # (nivel, texto, norm)
    for lv, text in seq:
        norm = text.lower().rstrip(":").strip() if lv else None
        if lv and out and out[-1][0] and out[-1][2] == norm:  # banner + encabezado duplicado
            if lv > out[-1][0]:
                out[-1] = (lv, text, norm)                    # conserva el más profundo (el real)
            continue
        out.append((lv, text, norm))
    return "\n".join(("#" * lv + " " + t) if lv else t for lv, t, _ in out)


DOC_TEXT = pdf_to_markdown(PDF_PATH)
heads = [ln for ln in DOC_TEXT.splitlines() if ln.startswith("#")]
print(f"{len(DOC_TEXT):,} chars · {len(heads)} encabezados\n")
print("\n".join(heads))


## 3. Ingest: chunking jerárquico + embeddings


In [ ]:
from sentence_transformers import SentenceTransformer


class STEmbedder(EmbeddingFunction):
    def __init__(self, model="sentence-transformers/all-MiniLM-L6-v2"):
        self.model_name, self.model = model, SentenceTransformer(model)
        self._dim = int(self.model.get_sentence_embedding_dimension())

    def __call__(self, input):
        return self.model.encode(input, normalize_embeddings=True).tolist()

    @staticmethod
    def name():
        return "st_notebook"

    def default_space(self):
        return "cosine"

    def supported_spaces(self):
        return ["cosine"]

    @staticmethod
    def build_from_config(c):
        return STEmbedder(c.get("model_name", "sentence-transformers/all-MiniLM-L6-v2"))

    def get_config(self):
        return {"model_name": self.model_name, "dim": self._dim}

    @staticmethod
    def validate_config(c):
        return None

    @property
    def dimension(self):
        return self._dim


db = GrafitoDatabase(":memory:")
db.create_vector_index("chunks", backend="bruteforce", embedding_function=STEmbedder(),
                       options={"metric": "cosine"})

ing = DocumentIngestor(db, chunker=MarkdownChunker(max_chars=1100, overlap=120),
                       embed_index="chunks", configure_fts=db.has_fts5(),
                       enricher=TitleContextEnricher(), hierarchy="auto")

DOC_KEY = "anthropic/building-effective-ai-agents"
result = ing.ingest(DOC_TEXT, document_key=DOC_KEY, title="Building Effective AI Agents",
                    source=PDF_URL, embed=True)
parent = db.match_nodes(labels=["Document"], properties={"document_key": DOC_KEY}, limit=1)[0]
print(f"sections={result.n_sections}  passages={result.n_passages}  hierarchy={result.hierarchy}")


## 4. Estructura: árbol de secciones (sin los chunks)


In [ ]:
struct_ids = {parent.id} | {
    n.id for n in db.match_nodes(properties={"managed_by": "grafito.document"})
    if n.properties.get("owner_document_id") == parent.id
    and ("Section" in n.labels or "DocumentVersion" in n.labels)
}
show_graph(db, struct_ids, title="Document → Version → Sections", height="540px")

print("ToC:")
for sec in ing.toc(DOC_KEY):
    print(f"  {sec.title}")
    for c in sec.children[:8]:
        print(f"     · {c.title}")


## 5. Búsqueda semántica + vecindario

Cada query dibuja **sólo** el hit, su sección/ancestros y los passages vecinos —
no el grafo entero.


In [ ]:
QUERIES = [
    "How do AI agents differ from traditional automation?",
    "When to use a single agent vs multi-agent orchestration?",
    "What are agent Skills and when to use them?",
    "customer support use cases for AI agents",
]


def neighborhood(hits, window=1):
    ids = {parent.id}
    for h in hits:
        ids.add(h.node.id)
        ex = ing.expand(h.node, window=window, include_ancestors=True)
        ids.update(p.id for p in ex.passages)
        ids.update(a.id for a in ex.ancestors)
        if ex.section:
            ids.add(ex.section.id)
    return ids


def run_query(query, k=3, draw=True):
    hits = ing.search(query, k=k)
    print(f"QUERY: {query}")
    for i, h in enumerate(hits, 1):
        print(f"  {i}. score={h.score:.3f} seq={h.global_seq}  "
              f"{(h.node.properties.get('text') or '')[:150].strip()}…")
    if draw and hits:
        show_graph(db, neighborhood(hits), title=query, hits={h.node.id for h in hits})
    return hits


run_query(QUERIES[0])


Probá otra (cambiá el índice):


In [ ]:
run_query(QUERIES[1], k=4)


## 6. Expand + pack (contexto para un LLM)


In [ ]:
hits = ing.search(QUERIES[1], k=3)
expanded = ing.expand(hits[0].node, window=1, include_parent=True, include_ancestors=True)
packed = ing.pack(expanded, max_chars=2000, include_citations=True)

print("section:", expanded.section and expanded.section.properties.get("title"))
print("ancestors:", [a.properties.get("title") for a in expanded.ancestors])
print("window seqs:", [p.properties.get("global_seq") for p in expanded.passages])
print("\n--- PACKED ---\n")
print(packed.text[:1800])

show_graph(db, neighborhood(hits[:1], window=1), title="Ventana de expand",
           hits={hits[0].node.id})


## 7. Hybrid search (vector + FTS + RRF)


In [ ]:
if db.has_fts5():
    for h in ing.hybrid_search(QUERIES[2], k=3):
        print(f"  rrf={h.score:.4f}  {(h.node.properties.get('text') or '')[:150].strip()}…")
else:
    print("FTS5 no disponible — se omite.")


## 8. Ejercicios

1. Corré `pdf_to_markdown` sobre otro PDF: no hay que tocar nada (headings por fuente).
2. `hierarchy=False` en un nuevo `DocumentIngestor` → grafo de chunks planos.
3. `window=0` vs `window=2` en `expand` sobre la misma query.
4. (Avanzado) `tree_select` con un LLM que elija `node_key` del ToC.

**Refs:** [Document Chunking](https://jpmanson.github.io/GrafitoDB/search/document-chunking/) ·
[Visualization](https://jpmanson.github.io/GrafitoDB/integrations/visualization/)


In [ ]:
db.close()
